# 01 - Exploratory Data Analysis

NHS Scotland A&E 4-hour compliance. This notebook documents the exploration that
**justifies the modelling decisions**: the structural break that shaped the split, the
data-quality findings behind the cleaning rules, and the train/holdout distribution
shift the model must extrapolate through.

Reproducible: run `python scripts/fetch_data.py` first, then run this notebook top to bottom.

In [ ]:
import sys; sys.path.insert(0, 'src')
import matplotlib.pyplot as plt
import pandas as pd
from ed_ops import data_quality as dq
from ed_ops.splits import build_temporal_split

panel = dq.build_primary_panel()
panel['m'] = panel['Month'].astype(int)
panel.shape, panel['TreatmentLocation'].nunique(), panel['HBT'].nunique()

## Panel profile
One row per (site x month), Type-1 EDs, AttendanceCategory 'All'. Compliance is
recomputed from counts (`within4 / total`), never the published percentage.

In [ ]:
print('rows      :', len(panel))
print('sites     :', panel['TreatmentLocation'].nunique())
print('boards    :', panel['HBT'].nunique())
print('months    :', panel['m'].min(), '->', panel['m'].max())
print('nulls     :', panel.isna().sum().sum())
panel['compliance_pct'].describe()

## The structural break (drives everything)
Compliance fell from ~97% (2007) to ~67% (2026) and is still declining. The dominant
break is 2022->2023, not COVID (2020->2022). A model trained on the old regime would
be biased high on the recent one, which is why the split trains on 2018+ only.

In [ ]:
annual = panel.assign(year=panel['m']//100).groupby('year')['compliance_pct'].median()
ax = annual.plot(marker='o', figsize=(9,4), color='#d62828')
ax.axhline(95, ls=':', color='grey'); ax.set_ylabel('median 4h compliance (%)')
ax.set_title('Annual median compliance - the structural break'); plt.tight_layout()

## Data-quality findings (F001-F005)
The cleaning rules are evidence-led. A few checks on the raw file that motivate them:

In [ ]:
raw = dq.load_activity_raw()
ci = dq.check_count_identity_activity(raw)
print('count identity failures (within4+over4 != total):', ci['identity_failures'])  # F005 ground truth
print('episode-null rows (F001, Type-3):', int(raw['NumberOfAttendancesEpisode'].isna().sum()))
print('duplicate-key rows (F002 G405H-201505):', dq.check_duplicate_keys_activity(raw)['rows_in_duplicate_groups'])
print('within4 > total rows (F003 W106H-202505):', dq.check_pct_bounds_activity(raw)['rows_invalid_counts_within_gt_total'])

## Train / validation / holdout distribution shift
The ~22pp gap between train (median ~89%) and holdout (median ~67%) is the break made
concrete: the model must **extrapolate**, not interpolate. The baselines face the same
gap, so the relative comparison stays fair.

In [ ]:
split = build_temporal_split(panel=panel)
parts = {'train': split.train.df, 'validation': split.validation.df, 'holdout': split.holdout.df}
plt.figure(figsize=(7,4))
plt.boxplot([p['compliance_pct'] for p in parts.values()], labels=list(parts))
plt.ylabel('4h compliance (%)'); plt.title('Compliance by partition'); plt.tight_layout()
{k: round(v['compliance_pct'].median(),1) for k,v in parts.items()}

## Takeaways for modelling
1. **Chronological split** is the only defensible design (forward-looking forecast on
   an autocorrelated time series).
2. **Persistence is a strong bar** because compliance is sticky month-to-month; seasonal
   naive is biased high by the downward trend.
3. The **structural break is the binding constraint**, not missing features - which is why
   the deliverable ensembles a tree with persistence rather than chasing a complex model.